# 04 — Where does STE fail? Ceiling + harder quantization + optimizer stress
**Re-upload the updated `qtta_lib.py`** to your `qtta-code` dataset first (new version; adds `CALIB` for min-max calibration).
Parts: A) TENT ceiling (FP32 / W8A8 / W4A4), B) harder quantizers, C) STE vs Gaussian under lr / batch-size stress.

In [ ]:
import sys, os, glob, copy, time
import numpy as np, pandas as pd, torch, torch.nn as nn
sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/**/qtta_lib.py", recursive=True)[0]))
import qtta_lib as Q
from qtta_lib import QCFG, CALIB
dev = "cuda" if torch.cuda.is_available() else "cpu"

CORR = ["gaussian_noise", "fog", "contrast"]          # 3 types: noise / weather / photometric
SEEDS = [0, 1]
SEV = 5
MEAN = torch.tensor(Q.MEAN).view(1, 3, 1, 1); STD = torch.tensor(Q.STD).view(1, 3, 1, 1)

def load_c(name, sev=SEV):
    path = glob.glob(f"/kaggle/input/**/{name}.npy", recursive=True)[0]
    lab = os.path.join(os.path.dirname(path), "labels.npy")
    sl = slice((sev - 1) * 10000, sev * 10000)
    x = torch.from_numpy(np.array(np.load(path, mmap_mode="r")[sl])).permute(0, 3, 1, 2).float() / 255
    y = torch.from_numpy(np.load(lab)[sl]).long()
    return ((x - MEAN) / STD).to(dev), y.to(dev)
DATA = {c: load_c(c) for c in CORR}

def entropy(lg): return -(lg.softmax(1) * lg.log_softmax(1)).sum(1).mean()

def configure(m):
    m.train()
    for p in m.parameters(): p.requires_grad_(False)
    params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.track_running_stats = False; mod.running_mean = None; mod.running_var = None
            mod.weight.requires_grad_(True); mod.bias.requires_grad_(True); params += [mod.weight, mod.bias]
    return params

def run_tta(model, x, y, mode="ste", sigma=0.5, lr=1e-3, bs=64, seed=0, adapt=True):
    m = copy.deepcopy(model); params = configure(m)
    QCFG.mode, QCFG.sigma = mode, sigma
    opt = torch.optim.Adam(params, lr=lr) if adapt else None
    perm = torch.randperm(len(x), generator=torch.Generator().manual_seed(seed)).to(dev)
    correct = 0
    with torch.set_grad_enabled(adapt):
        for i in range(0, len(x), bs):
            idx = perm[i:i + bs]
            logits = m(x[idx])
            if adapt:
                opt.zero_grad(set_to_none=True); entropy(logits).backward(); opt.step()
            correct += (logits.argmax(1) == y[idx]).sum().item()
    return 100 * correct / len(x)

@torch.no_grad()
def static_acc(model, x, y):
    m = copy.deepcopy(model).eval()
    return sum((m(x[i:i + 500]).argmax(1) == y[i:i + 500]).sum().item() for i in range(0, len(x), 500)) / len(x) * 100

fp, _ = Q.load_fp32(Q.find_file("resnet20_fp32.pt"), dev)
train_loader, test_loader = Q.get_cifar10(bs=500)

def build(wb, ab, minmax=False):
    CALIB.act, CALIB.w = ((1.0, 1.0, 1), (1.0, 1.0, 1)) if minmax else ((0.3, 1.0, 36), (0.4, 1.0, 31))
    m, _ = Q.quantize_model(fp, wb, ab, train_loader, dev)
    CALIB.act, CALIB.w = (0.3, 1.0, 36), (0.4, 1.0, 31)
    return m

MODELS = {"FP32": fp, "W8A8": build(8, 8), "W4A4": build(4, 4),
          "W3A3": build(3, 3), "W4A4-minmax": build(4, 4, minmax=True)}
for n, m in MODELS.items(): print(f"{n:12s} clean acc = {Q.evaluate(m, test_loader, dev):.2f}")

## Part A/B — ceiling and harder quantizers (no-TTA, BN-stats only, STE-TENT), default lr=1e-3, bs=64

In [ ]:
rows = []
for n, m in MODELS.items():
    for c in CORR:
        x, y = DATA[c]
        rows.append(dict(model=n, corruption=c, no_TTA=static_acc(m, x, y),
                         BN_only=run_tta(m, x, y, adapt=False),
                         STE=np.mean([run_tta(m, x, y, "ste", seed=s) for s in SEEDS])))
A = pd.DataFrame(rows).groupby("model")[["no_TTA", "BN_only", "STE"]].mean().loc[list(MODELS)]
A["TTA_gain_over_BN"] = A["STE"] - A["BN_only"]
A["gap_to_FP32_TENT"] = A.loc["FP32", "STE"] - A["STE"]
print(A.round(2).to_string())

## Part C — STE vs Gaussian under optimizer stress
Settings: (lr, batch size). Methods: STE and Gaussian sigma/Delta in {0.35, 1.0, 2.0}.

In [ ]:
SETTINGS = [(1e-3, 64), (1e-2, 64), (1e-3, 16), (1e-2, 16)]
METHODS = [("STE", "ste", 0.0), ("g0.35", "gauss", 0.35), ("g1.0", "gauss", 1.0), ("g2.0", "gauss", 2.0)]
SCAN_MODELS = ["W4A4", "W4A4-minmax", "W3A3"]
res = []
for mn in SCAN_MODELS:
    for lr, bs in SETTINGS:
        for c in CORR:
            x, y = DATA[c]
            for name, mode, s in METHODS:
                for seed in SEEDS:
                    res.append(dict(model=mn, lr=lr, bs=bs, corruption=c, method=name, seed=seed,
                                    acc=run_tta(MODELS[mn], x, y, mode, s, lr=lr, bs=bs, seed=seed)))
        pd.DataFrame(res).to_csv("/kaggle/working/scan.csv", index=False)
        print("done", mn, lr, bs, flush=True)

In [ ]:
df = pd.DataFrame(res)
T = df.groupby(["model", "lr", "bs", "method"]).acc.mean().unstack("method")[[n for n, _, _ in METHODS]]
for n, _, _ in METHODS[1:]: T[f"d_{n}"] = T[n] - T["STE"]
print(T.round(2).to_string())

# paired check: per (model,lr,bs,corruption,seed) difference vs STE, how often does Gaussian win?
P = df.pivot_table(index=["model", "lr", "bs", "corruption", "seed"], columns="method", values="acc")
print("\nwin rate vs STE (fraction of paired runs):")
print(pd.DataFrame({n: (P[n] > P["STE"]).groupby(["model", "lr", "bs"]).mean() for n, _, _ in METHODS[1:]}).round(2).to_string())